<a href="https://colab.research.google.com/github/someonesomeon/aig230_lab01_atong23/blob/main/AIG230_NLP_Midterm_March_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIG230 NLP Midterm - March 2nd 2026
All answers must be computed using code. Provide numeric outputs and short interpretations.
## Instructions
- You must compute answers using code.
- Many questions require numeric answers.
- Interpretation must be supported by computed results.

In [2]:
!pip install gensim
import nltk, string, math, random, re
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
print('All imports successful')
print(f'TensorFlow version: {tf.__version__}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 61.0 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


All imports successful
TensorFlow version: 2.19.0


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


# Load Corpora

In [3]:
# Load corpora
with open('corpus_tech_ai_labeled.csv', 'r', encoding='utf-8') as f:
    tech_text = f.read()

with open('corpus_movie_reviews_labeled.csv', 'r', encoding='utf-8') as f:
    review_text = f.read()

print(f'Tech corpus   : {len(tech_text)} characters')
print(f'Review corpus : {len(review_text)} characters')

Tech corpus   : 13049 characters
Review corpus : 4044 characters


## Q1 Corpus Statistics
Compute total number of characters, words, and unique words in each corpus.
What is the vocabulary size of each corpus?

In [12]:

total_characters = len(tech_text)
tech_tokens = re.findall(r"\b\w+\b", tech_text.lower())
total_words = len(words)
unique_words = len(set(words))
tech_vocab = set(tech_tokens)


print("Total Characters:", total_characters)
print("Total Words:", total_words)
print("Unique Words:", unique_words)
print("Vocabulary Size:", tech_vocab)

total_characters = len(review_text)
review_tokens = re.findall(r"\b\w+\b", review_text.lower())
total_words = len(words)
unique_words = len(set(words))
review_vocab = set(review_tokens)


print("Total Characters:", total_characters)
print("Total Words:", total_words)
print("Unique Words:", unique_words)
print("Vocabulary Size:", review_vocab)

Total Characters: 13049
Total Words: 637
Unique Words: 85
Vocabulary Size: {'deployment', 'chatbots', 'natural', 'assist', 'neural', 'processing', 'analysis', 'detect', 'responsible', 'perception', 'world', 'to', 'atmospheric', 'sparsity', 'integrates', 'flow', 'simulation', 'risk', 'accountability', 'scientists', 'statistical', 'diseases', 'features', 'between', 'power', 'are', 'large', 'projections', 'outcomes', 'similarity', 'ai', 'agents', 'model', 'rather', 'use', 'energy', 'hospitals', 'machine', 'in', 'fairness', 'accelerate', 'data', 'handcrafted', 'practical', 'readings', 'enables', 'networks', 'healthcare', 'estimate', 'medical', 'tokens', 'grid', 'extreme', 'terms', 'control', 'capture', 'environments', 'governments', 'summarization', 'ethical', 'classify', 'semantic', 'optimization', 'learning', 'propose', 'planning', 'text', 'climate', 'imaging', 'carefully', 'label_numeric', 'privacy', 'relies', 'word', '1', 'high', 'context', 'deep', 'treatments', 'than', 'learned', 'ben

In [13]:
print('=' * 55)
print(f'{"Metric":<30} {"Tech":>10} {"Reviews":>10}')
print('=' * 55)
print(f'{"Characters":<30} {len(tech_text):>10,} {len(review_text):>10,}')
print(f'{"Total tokens (words)":<30} {len(tech_tokens):>10,} {len(review_tokens):>10,}')
print(f'{"Vocabulary size (unique words)":<30} {len(tech_vocab):>10,} {len(review_vocab):>10,}')
print('=' * 55)

Metric                               Tech    Reviews
Characters                         13,049      4,044
Total tokens (words)                1,667        637
Vocabulary size (unique words)        149         85


### Answer Q1:


The vocabulary size of the tech corpus = 149, and vocab size of review corpus  = 85

## Q2 Lexical Diversity
Tokenize and lowercase both corpora. Compute type-token ratio.
Which corpus is more lexically diverse?

In [23]:
tech_tokens = re.findall(r"\b\w+\b", tech_text.lower())
review_tokens = re.findall(r"\b\w+\b", review_text.lower())

tech_types = len(set(tech_tokens))
review_types = len(set(review_tokens))

tech_ttr = tech_types / len(tech_tokens)
review_ttr = review_types / len(review_tokens)

print("Tech TTR:", round(tech_ttr, 4))
print("Reviews TTR:", round(review_ttr, 4))

print(len(tech_tokens))

Tech TTR: 0.0894
Reviews TTR: 0.1334
1667
149


### Answer Q2:


Reviews corpus is more lexically diverse

## Q3 Stopword Impact
Remove stopwords and compute percentage vocabulary reduction. How much does vocabulary size decrease (percentage) for each corpus?

In [44]:
stop_words = set(stopwords.words('english'))
# Continue
#assuming unique or not unique?
tech_tokens = re.findall(r"\b\w+\b", tech_text.lower())
review_tokens = re.findall(r"\b\w+\b", review_text.lower())

tech_nosw = [w for w in tech_tokens if w not in stop_words]
review_nosw = [w for w in review_tokens if w not in stop_words]


tech_vocab_original = len(set(tech_nosw))
review_vocab_original = len(set(review_nosw))


tech_filtered = [w for w in tech_tokens if w not in stop_words]
review_filtered = [w for w in review_tokens if w not in stop_words]

tech_vocab_filtered = len(set(tech_filtered))
review_vocab_filtered = len(set(review_filtered))


tech_reduction = ((tech_vocab_original - tech_vocab_filtered) / tech_vocab_original) * 100
review_reduction = ((review_vocab_original - review_vocab_filtered) / review_vocab_original) * 100

print(f"Tech vocabulary reduced by {tech_reduction:.2f}%")
print(f"Reviews vocabulary reduced by {review_reduction:.2f}%")

Tech vocabulary reduced by 0.00%
Reviews vocabulary reduced by 0.00%


In [22]:
# Use this formula to calculate percentage of reduction
def pct_reduction(before, after):
    return 100 * (1 - len(after) / len(before))

In [ ]:
tech_percentage = pct_reduction()
review_percentage = pct_reduction()

### Answer Q3:


## Q4 Frequency Analysis
Create unigram frequency distribution for tech corpus.
What are the top 10 most frequent words?

In [28]:
#including stopwords?
freq_dist = Counter(tech_filtered)
top_10 = freq_dist.most_common(10)
print("Top 10 most frequent words in Tech corpus:")
for word, count in top_10:
    print(f"{word}: {count}")

Top 10 most frequent words in Tech corpus:
neutral: 144
1: 144
systems: 24
learning: 24
models: 24
model: 16
neural: 16
networks: 16
remain: 16
language: 16


### Answer Q4:


Top 10 most frequent words in Tech corpus:

neutral: 144

1: 144

systems: 24

learning: 24

models: 24

model: 16

neural: 16

networks: 16

remain: 16

language: 16

## Q5 Bigram Counts
Build bigram model for tech corpus.
How many unique bigrams exist in tech corpus?

In [30]:
# Using zip
bigrams = list(zip(tech_filtered, tech_filtered[1:]))
unique_bigrams = set(bigrams)
num_unique_bigrams = len(unique_bigrams)

print("Number of unique bigrams in Tech corpus:", num_unique_bigrams)

Number of unique bigrams in Tech corpus: 163


### Answer Q5:


Number of unique bigrams in Tech corpus: 163

## Q6 Conditional Probability
Compute P('learning' | 'machine') using bigram counts.

Formula: P(learning | machine) = count('machine learning') / count('machine')

In [38]:
token_counts = Counter(tech_filtered)
bigram_counts = Counter(bigrams)

count_machine_learning = bigram_counts[('machine', 'learning')]  # count of 'machine learning'
count_machine = token_counts['machine']         # count of 'machine'

if count_machine > 0:
    prob = count_machine_learning / count_machine
else:
    prob = 0


In [39]:
print(f"count('machine learning') = {count_machine_learning}")
print(f"count('machine')          = {count_machine}")
print(f"\nP('learning' | 'machine') = {count_machine_learning}/{count_machine} = {prob:.4f}")

count('machine learning') = 8
count('machine')          = 8

P('learning' | 'machine') = 8/8 = 1.0000


### Answer Q6:


100% for probability using count of machine learning/machine

## Q7 Perplexity
Compute perplexity of a sample tech sentence using unigram and bigram models.

Formulas:

Unigram: PP = exp(-(1/N) * sum(log P(wi)))

Bigram: PP = exp(-(1/(N-1)) * sum(log P(wi | wi-1)))



In [41]:
sample_sentence = 'Large language models generate responses by predicting tokens sequentially'

In [42]:
# Unigram Perplexity

sample_tokens = re.findall(r"\b\w+\b", sample_sentence.lower())
N = len(sample_tokens)


unigram_counts = Counter(tech_tokens)
total_tokens = len(tech_tokens)

# Compute probability of each word in sample
unigram_probs = []
for w in sample_tokens:
    prob = unigram_counts.get(w, 0) / total_tokens  # If word unseen, prob=0
    # Avoid log(0) by using a small smoothing value
    if prob == 0:
        prob = 1e-6
    unigram_probs.append(prob)

# Compute Unigram Perplexity
pp_unigram = math.exp(- (1/N) * sum(math.log(p) for p in unigram_probs))
print(f"Unigram Perplexity: {pp_unigram:.4f}")

Unigram Perplexity: 170.7593


In [43]:
# Bigram Perplexity

sample_bigrams = list(zip(sample_tokens, sample_tokens[1:]))

bigram_probs = []
for w1, w2 in sample_bigrams:
    count_bigram = bigram_counts.get((w1, w2), 0)
    count_w1 = token_counts.get(w1, 0)
    # Add simple smoothing to avoid zero probability
    prob = (count_bigram + 1) / (count_w1 + len(unigram_counts))
    bigram_probs.append(prob)

# Compute Bigram Perplexity
pp_bigram = math.exp(- (1/(N-1)) * sum(math.log(p) for p in bigram_probs))
print(f"Bigram Perplexity: {pp_bigram:.4f}")

Bigram Perplexity: 30.5735


In [ ]:
print(f'Sample : "{sample_sentence}"')
print(f'Tokens : {sample_tokens}')
print(f'N      : {N}')
print(f'\nUnigram Perplexity : {perplexity_unigram:.2f}')
print(f'Bigram  Perplexity : {perplexity_bigram:.2f}')
print(f'Ratio (uni/bi)     : {perplexity_unigram/perplexity_bigram:.1f}x reduction with bigram')

### Answer Q7:


Unigram Perplexity:
Bigram Perplexity: 30.

## Q8 Word2Vec
Train Word2Vec on movie reviews corpus.
What is the vocabulary size? What is vector dimension? What word is most similar to 'visuals'?

NOTE: Use min_count=1 (required for small corpus) and vector_size=20.

In [50]:
model = Word2Vec(
    review_filtered,
    vector_size=20,  # as specified
    window=5,
    min_count=1,     # required for small corpus
    workers=1,       # number of CPU threads
    sg=1             # skip-gram (optional; default CBOW is 0)
)

vocab_size = len(model.wv.index_to_key)  # number of unique words
vector_dim = model.vector_size

print("Vocabulary size:", vocab_size)
print("Vector dimension:", vector_dim)


if 'visuals' in model.wv:
    most_similar = model.wv.most_similar('visuals', topn=1)
    print("Most similar word to 'visuals':", most_similar[0])
else:
    print("'visuals' is not in the vocabulary")

Vocabulary size: 27
Vector dimension: 20
'visuals' is not in the vocabulary


### Answer Q8:


WARNING:gensim.models.word2vec:Each 'sentences' item should be a list of words (usually unicode strings). First item here is instead plain <class 'str'>.
Vocabulary size: 27
Vector dimension: 20
'visuals' is not in the vocabulary

## Q9 Naive Bayes
Train Naive Bayes on movie corpus (positive vs negative). Mixed sentences excluded.
Report accuracy.

In [51]:
import csv

# --- Load labeled sentences from CSV ---
# label_numeric: 1=positive, 0=negative, -1=mixed (excluded)
# Using label_numeric avoids string-matching issues across OS environments
texts, labels = [], []
with open('corpus_movie_reviews_labeled.csv', 'r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        numeric = int(row['label_numeric'])
        if numeric in (1, 0):           # skip mixed (-1)
            texts.append(row['text'].strip())
            labels.append(numeric)

print(f'Samples loaded : {len(texts)}')
print(f'  Positive (1) : {labels.count(1)}')
print(f'  Negative (0) : {labels.count(0)}')
print(f'  Mixed        : excluded (label_numeric=-1)')

if len(texts) == 0:
    raise FileNotFoundError(
        'No samples loaded. Make sure corpus_movie_reviews_labeled.csv '
        'is in the same folder as this notebook.')

Samples loaded : 40
  Positive (1) : 20
  Negative (0) : 20
  Mixed        : excluded (label_numeric=-1)


In [53]:
# --- TF-IDF features ---
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

Train samples: 32, Test samples: 8


In [54]:
# --- Train / test split (80/20, stratified) ---
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(lowercase=True, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [56]:
# --- Train Naive Bayes ---
nb = MultinomialNB()
# Continue coding

nb.fit(X_train_vec, y_train)


MultinomialNB()

### Answer Q9:


## Q10 Precision/Recall/F1
Report precision, recall, F1.

In [65]:
from sklearn.metrics import accuracy_score

prec_nb = nb.predict(X_test_vec)
nb_acc = accuracy_score(y_test, y_pred)

print(f"Naive Bayes Accuracy: {accuracy:.4f}")

from sklearn.metrics import classification_report
report = classification_report(y_test, y_pred, target_names=['Negative', 'Positive'])
print(report)

Naive Bayes Accuracy: 1.0000
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00         4
    Positive       1.00      1.00      1.00         4

    accuracy                           1.00         8
   macro avg       1.00      1.00      1.00         8
weighted avg       1.00      1.00      1.00         8



In [66]:
print('Naive Bayes -- Classification Report')
print('=' * 40)
print(f'  Accuracy  : {nb_acc:.4f}')
print(f'  Precision : {prec_nb:.4f}')
print(f'  Recall    : {rec_nb:.4f}')
print(f'  F1-Score  : {f1_nb:.4f}')

Naive Bayes -- Classification Report
  Accuracy  : 1.0000


TypeError: unsupported format string passed to numpy.ndarray.__format__

### Answer Q10:


## Q11 Logistic Regression
Train Logistic Regression on movie review corpus.
Which model performs better? Compare performance numerically.

In [52]:
# train logistic regression
lr = LogisticRegression(max_iter=1000, random_state=42)
# Continue coding

In [69]:

lr.fit(X_train_vec, y_train)

from sklearn.metrics import accuracy_score, classification_report

# Predict on test set
y_pred_lr = lr.predict(X_test_vec)

# Accuracy
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"Logistic Regression Accuracy: {accuracy_lr:.4f}")

# Detailed report (precision, recall, f1-score)
print(classification_report(y_test, y_pred_lr, target_names=['Negative', 'Positive']))

Logistic Regression Accuracy: 1.0000
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00         4
    Positive       1.00      1.00      1.00         4

    accuracy                           1.00         8
   macro avg       1.00      1.00      1.00         8
weighted avg       1.00      1.00      1.00         8



### Answer Q11:


Logistic Regression Accuracy: 1.0000
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00         4
    Positive       1.00      1.00      1.00         4

    accuracy                           1.00         8
   macro avg       1.00      1.00      1.00         8
weighted avg       1.00      1.00      1.00         8

## Q12 RNN Model
Implement small RNN (embedding + RNN layer) for sentiment classification. Report training accuracy.

In [72]:
# Preprocessing

tokenizer_rnn = keras.preprocessing.text.Tokenizer()
tokenizer_rnn.fit_on_texts(texts)
sequences = tokenizer_rnn.texts_to_sequences(texts)

VOCAB_SIZE = len(tokenizer_rnn.word_index) + 1
MAX_LEN    = max(len(s) for s in sequences)

X_rnn = keras.preprocessing.sequence.pad_sequences(sequences, maxlen=MAX_LEN, padding='post')
y_rnn = np.array(labels)

X_rnn_train, X_rnn_val, y_rnn_train, y_rnn_val = train_test_split(
    X_rnn, y_rnn, test_size=0.20, random_state=42, stratify=y_rnn)

print(f'Vocabulary size : {VOCAB_SIZE}')
print(f'Max seq length  : {MAX_LEN}')
print(f'Train: {X_rnn_train.shape[0]}   Val: {X_rnn_val.shape[0]}')

tf.random.set_seed(42)

Vocabulary size : 57
Max seq length  : 10
Train: 32   Val: 8


In [75]:
# RNN Implementation
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    SimpleRNN(64, activation='tanh'),
    Dense(1, activation='sigmoid')  # binary classification
])

rnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

rnn_model.summary()


NameError: name 'EMBEDDING_DIM' is not defined

In [ ]:
# Model compile
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Model training
history = model.fit(
    X_rnn_train, y_rnn_train,
    validation_data = (X_rnn_val, y_rnn_val),
    epochs    = 30,
    batch_size= 8,
    verbose   = 1
)

### Answer Q12:


## Q13 Overfitting Check
Compare train vs validation accuracy.
Does RNN overfit? Justify using loss or accuracy trends.
Plot train vs validation accuracy and loss.

### Answer Q13:


## Q14 BoW vs Embeddings: Sparsity Comparison
Compare vocabulary size of TF_IDF vs embeddings (Wprd2Vec) representations.
Explain sparsity difference numerically.

Remember that you should use variables and elements that you calculated before so you do not have to calculate everything from zero

### Answer Q14:


# Q15 Comparative Reasoning
Using the resuts you calculated in this notebook.
Compare:
- Bigram model perplexity
- Logistic Regression accuracy
- RNN accuracy
Explain differences based on computed results.

### Answer Q15:
